# D&D Session Transcription + Speaker Diarization (whisperX)

Run on Colab GPU

In [3]:
from datetime import datetime
import os

# os.environ["HF_TOKEN"] = ""
# HF_TOKEN = os.environ["HF_TOKEN"]

In [4]:
!pip install whisperx -q

In [5]:
from google.colab import drive, files
drive.mount('/content/drive')

# then point to your file
audio_file = '/content/drive/MyDrive/<>.wav'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Transcribe + Align

In [6]:
import whisperx
import torch

if torch.cuda.is_available():
    print("CUDA is available")
    device = "cuda"
else:
    print("CUDA is not available")
    exit()

CUDA is available


In [7]:
compute_type = "float16"
batch_size = 16

model = whisperx.load_model("medium", device=device, compute_type=compute_type, language="en")
audio = whisperx.load_audio(audio_file)
result = model.transcribe(audio, batch_size=batch_size)

print(f"Transcribed {len(result['segments'])} segments")

align_model, metadata = whisperx.load_align_model(language_code="en", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device=device)
print(result["segments"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


2026-06-05 01:08:05 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
/usr/local/lib/python3.12/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issue

Transcribed 472 segments
{'start': 0.031, 'end': 4.152, 'text': ' what the others see.', 'words': [{'word': 'what', 'start': 0.031, 'end': 0.651, 'score': np.float64(0.61)}, {'word': 'the', 'start': 2.672, 'end': 2.752, 'score': np.float64(0.959)}, {'word': 'others', 'start': 2.812, 'end': 3.252, 'score': np.float64(0.823)}, {'word': 'see.', 'start': 3.852, 'end': 4.152, 'score': np.float64(0.974)}], 'avg_logprob': -0.3021399417649145}


## Diarize

In [8]:
from whisperx.diarize import DiarizationPipeline
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)
diarize_segments = diarize_model(audio)
result = whisperx.assign_word_speakers(diarize_segments, result)
print(diarize_segments)
print(result["segments"][1])

2026-06-05 01:16:50 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1839.)
  std = sequences.std(dim=-1, correction=1)


                                segment label     speaker         start  \
0     [ 00:00:00.030 -->  00:00:04.182]     A  SPEAKER_00      0.030969   
1     [ 00:00:06.494 -->  00:00:07.709]     E  SPEAKER_04      6.494094   
2     [ 00:00:10.004 -->  00:00:15.910]     E  SPEAKER_04     10.004094   
3     [ 00:00:13.935 -->  00:00:14.222]     A  SPEAKER_00     13.935969   
4     [ 00:00:16.534 -->  00:00:16.939]     A  SPEAKER_00     16.534719   
...                                 ...   ...         ...           ...   
2641  [ 03:23:09.788 -->  03:23:10.311]     A  SPEAKER_00  12189.788469   
2642  [ 03:23:11.594 -->  03:23:12.336]     A  SPEAKER_00  12191.594094   
2643  [ 03:23:13.720 -->  03:23:19.508]     A  SPEAKER_00  12193.720344   
2644  [ 03:23:20.217 -->  03:23:21.752]     A  SPEAKER_00  12200.217219   
2645  [ 03:23:22.866 -->  03:23:25.313]     A  SPEAKER_00  12202.866594   

               end  
0         4.182219  
1         7.709094  
2        15.910344  
3        14.222

In [9]:
print(result["segments"][2])

{'start': 16.635, 'end': 16.915, 'text': 'Okay.', 'words': [{'word': 'Okay.', 'start': 16.635, 'end': 16.915, 'score': np.float64(0.489), 'speaker': 'SPEAKER_00'}], 'avg_logprob': -0.3021399417649145, 'speaker': 'SPEAKER_00'}


## Format & Save Output

In [10]:
with open("transcription_diarized.txt", "w") as f:
    for segment in result["segments"]:
        if "speaker" not in segment:
            segment["speaker"] = "Unknown"
        line = f"Speaker {segment['speaker']} from {segment['start']:.2f} to {segment['end']:.2f}: {segment['text']}"
        print(line)
        f.write(line + "\n")

In [11]:
with open("/content/drive/MyDrive/transcription_diarized.txt", "w") as f:
    for segment in result["segments"]:
        if "speaker" not in segment:
            segment["speaker"] = "Unknown"
        line = f"Speaker {segment['speaker']} from {segment['start']:.2f} to {segment['end']:.2f}: {segment['text']}"
        f.write(line + "\n")

print("Saved to Google Drive!")

# download to local machine
files.download("/content/drive/MyDrive/transcription_diarized.txt")

Saved to Google Drive!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>